# ODI to Databricks Migration: trg_loc_load

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook performs an initial load of the `trg_loc` table in the `HR` schema from the `LOCATIONS` source table.

**Source ODI Scenarios:**
- SCEN_TASK_NO {10}: (No SQL provided)
- SCEN_TASK_NO {20}: (No SQL provided)
- SCEN_TASK_NO {30}: INSERT into HR.TRG_LOC from HR.LOCATIONS

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

# ETL Parameters

The following parameters are used to control the ETL process and are typically set by the orchestrator.

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
    '${ETL_PROC_WID}' AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

# Target Table: `workspace.hr.trg_loc`

This section defines and loads data into the permanent target table.

In [ ]:
%sql
-- Create target table if it doesn't exist
CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc (
    LOCATION_ID     BIGINT,
    STREET_ADDRESS  STRING,
    POSTAL_CODE     STRING,
    CITY            STRING,
    STATE_PROVINCE  STRING,
    COUNTRY_ID      STRING
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Insert into target table
INSERT INTO workspace.hr.trg_loc (
    LOCATION_ID,
    STREET_ADDRESS,
    POSTAL_CODE,
    CITY,
    STATE_PROVINCE,
    COUNTRY_ID
)
SELECT
    LOCATIONS.LOCATION_ID,
    LOCATIONS.STREET_ADDRESS,
    LOCATIONS.POSTAL_CODE,
    LOCATIONS.CITY,
    LOCATIONS.STATE_PROVINCE,
    LOCATIONS.COUNTRY_ID
FROM
    workspace.hr.locations AS LOCATIONS;

In [ ]:
%sql
-- Record count after insert
SELECT COUNT(*) FROM workspace.hr.trg_loc;

# Cleanup

No temporary C$ or I$ tables were created for this flow, so no cleanup is required here.

# Validation

Verify the data in the target table after the load.

In [ ]:
%sql
-- Display total rows in the target table
SELECT COUNT(*) AS total_rows FROM workspace.hr.trg_loc;

In [ ]:
%sql
-- Sample records from the target table to verify data
SELECT *
FROM workspace.hr.trg_loc
ORDER BY LOCATION_ID
LIMIT 100;

# Conversion Notes

1.  **Schema and Table Naming:** All Oracle schema references (`HR`) have been converted to `workspace.hr` and table names (`TRG_LOC`, `LOCATIONS`) to lowercase (`trg_loc`, `locations`).
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable to Databricks Delta Lake.
3.  **Data Types:** Inferred Oracle `NUMBER(4)` for `LOCATION_ID` has been mapped to `BIGINT` in Spark SQL for safety. `VARCHAR2` and `CHAR` types have been mapped to `STRING`.
4.  **Load Type:** The original ODI SQL performs a direct `INSERT INTO ... SELECT FROM ...`, indicating a full load or initial population. No incremental logic (e.g., `WHERE EXISTS` for updates or `WHERE NOT EXISTS` for inserts) was present in the provided snippet.
5.  **No Staging/Flow Tables:** This conversion did not require `C$` or `I$` staging/flow tables based on the input SQL, which directly loads into the target.